# Week 49, Day 1: Brain networks as graphs: small worlds

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The small-world transition

Write clustering(A) using the diagonal of A cubed, and pathlen(A) using scipy's shortest_path on the unweighted graph. Compute both, relative to the lattice, for rewiring probabilities 0.001, 0.01, 0.1 and 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import shortest_path
r = np.random.default_rng(1998)

def ws(N=200, k=6, p=0.0):
    A = np.zeros((N, N), bool)
    for i in range(N):
        for j in range(1, k // 2 + 1):
            A[i, (i + j) % N] = A[(i + j) % N, i] = True
    for i in range(N):
        for j in range(1, k // 2 + 1):
            t = (i + j) % N
            if r.random() < p and A[i, t]:
                n = r.choice(np.where(~A[i] & (np.arange(N) != i))[0])
                A[i, t] = A[t, i] = False; A[i, n] = A[n, i] = True
    return A

def clustering(A):
    ...

def pathlen(A):
    ...

A0 = ws(); C0, L0 = clustering(A0), pathlen(A0)
res = {}
for p in [0.001, 0.01, 0.1, 1.0]:
    A = ws(p=p); res[p] = (clustering(A) / C0, pathlen(A) / L0)
    print(f"p = {p:5.3f}: C/C0 = {res[p][0]:.3f}, L/L0 = {res[p][1]:.3f}")
ps = list(res); plt.semilogx(ps, [res[p][0] for p in ps], "o-", label="C(p)/C(0)"); plt.semilogx(ps, [res[p][1] for p in ps], "s-", label="L(p)/L(0)")
plt.xlabel("rewiring probability"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_window():
    "at p = 0.01 clustering is above 90% of the lattice's while path length is below 70%"
    assert res[0.01][0] > 0.9 and res[0.01][1] < 0.7
def test_random():
    "at p = 1 clustering falls below 10% of the lattice's"
    assert res[1.0][0] < 0.1

test_window()
test_random()
print("All checks passed")

### Exercise: How small is this world?

Use a fully rewired network (p = 1) as the random baseline. Compute sigma for p = 0, 0.05 and 1.

In [ ]:
Ar = ws(p=1.0); C_rand_sim, L_rand_sim = clustering(Ar), pathlen(Ar)
print(f"random baseline: C = {C_rand_sim:.3f} (k/N = {6 / 200:.3f}), L = {L_rand_sim:.2f} (ln N / ln k = {np.log(200) / np.log(6):.2f})")
sig = {}
for p in [0.0, 0.05, 1.0]:
    A = ws(p=p)
    sig[p] = ...
    print(f"p = {p}: small-world index sigma = {sig[p]:.2f}")

In [ ]:
import numpy as np
def test_theory():
    "the fully rewired graph's clustering and path length are within 50% and 15% of the random-graph formulas"
    assert abs(C_rand_sim / (6 / 200) - 1) < 0.5 and abs(L_rand_sim / (np.log(200) / np.log(6)) - 1) < 0.15
def test_sigma():
    "the p = 0.05 network's index is more than twice the lattice's, and the random graph's is near 1"
    assert sig[0.05] > 2 * sig[0.0] and sig[1.0] < 1.5

test_theory()
test_sigma()
print("All checks passed")